In [1]:
import os
# os.sysjupyter nbextension enable --py --sys-prefix widgetsnbextension
from IPython.display import HTML
HTML('''<script>
code_show=true;
function code_toggle() {
 if (code_show){
 $('div.input').hide();
 } else {
 $('div.input').show();
 }
 code_show = !code_show
}
$( document ).ready(code_toggle);
</script>
The raw code for this IPython notebook is by default hidden for easier reading.
To toggle on/off the raw code, click <a href="javascript:code_toggle()">here</a>.''')

# Fourier series solution to Terzaghi's one-dimensional consolidation equation¶

## Developed by: Scott Brandenberg (sjbrandenberg@g.ucla.edu)
## Modified by: Pedro Arduino (parduino@uw.edu)

There are a variety of ways to solve second orde differential equations of the form of Terzaghis 1D consolidation equation.
Some are mthematically exact, others are only approximate. Taylor (1948) provides a mathematically rigorous solution in terms of Fourier series expansion The development that follows uses this approach.

## 1. Assumptions

<ol>
<li>Loading is one-dimensional. Settlement and flow of water is vertical.</li>
<li>Compressibility is constant</li>
<li>Permeability is constant</li>
<li>Flow is controlled by Darcy's law</li>
<li>Secondary compression does not occur</li>
<li>Deformations are small, such that strains may be computed from undeformed geoemtry</li>
    <li> Drainage allowed along both boundaries</li>
<li>Soil is saturated and uniform</li>
</ol>

<figure>
<img src="Consolidation.png">
<figcaption><strong>Figure 1. </strong>Compressible soil and assumptions made in Terzaghi's one-dimensional consolidation theory.</figcaption>
</figure>

Terzaghi's one-dimensional consolidation equation is:

$\qquad c_v\frac{\partial^2 u}{\partial z^2} = \frac{\partial u}{\partial t}$

The solution to this equation is obtained by separation of variables and a Fourier series to be:

$\qquad U_z = 1 - \sum_{m=0}^{\infty} \frac{2}{M} sin\left(M Z \right)e^{-M^2 T}$

$\qquad$ where $M = \frac{\pi}{2} (2m+1)$,

$\qquad$ and $U_z = 1 - \frac{u_e}{\Delta\sigma_v}$, $T = \frac{c_v t}{H^2}$, $Z = \frac{z}{H}$, $z$ is depth, and $H$ is drainage path length.

Of course, we can't actually perform this sum from 1 to $\infty$, so we have to select a suitable number of terms in the Fourier series, N, to obtain an accurate solution:

$\qquad U_z = 1 - \sum_{m=0}^{N} \frac{2}{M} sin\left(M Z \right)e^{-M^2 T}$

The question is how many terms do we need to include in the Fourier series to obtain an accurate solution? Run the code cell below, and move the slider to observe the influence of N on the solution. Feel free to alter the list of T values if you'd like to explore.

You should find that the solutions for $T$ larger than about 0.1 are relatively insensitive to $N$. However, the number of terms in the Fouier series has a significant effect for low values of $T$. This is because the solution at low $T$ exhibits a flat region at $U_z = 0$, and significantly more terms are required to obtain an accurate series solution. Using a single term provides a reasonable approximation when $T$ is higher than about 0.2.

In [2]:
%matplotlib notebook

import ipywidgets as widgets
import numpy as np
from matplotlib import pyplot as plt

def Uz(N,Z,T):
    uz = 0
    #function is zero for even values of n. Therefore only compute for odd values
    for m in range(0,N,1):
        M = (2*m+1)*np.pi/2
        uz = uz + 2.0/M * np.sin(M*Z)*np.exp(-1.0*M**2*T)
    return 1.0-uz

T = [0.01,0.05,0.1,0.2,0.3,0.4,0.5,0.6,0.7,0.8,0.9, 0.5]
Z = np.linspace(0.0,2.0,num=50)
N = 19

fig,ax = plt.subplots(1,1)
ax.set_xlabel('$U_z$')
ax.set_ylabel('$Z$')
ax.set_xlim(-0.2,1.0)
ax.set_ylim(0,2)
ax.grid(True)
#ax.legend()
plots = []
for j, t in enumerate(T):
    u=[]
    for i, z in enumerate(Z):
        u.append(Uz(N,z,t))

    if (j < 11):
        plt.plot(u,Z, \
            color='blue', linestyle='dashed', linewidth=0.7, label = 'T')
    if (j == 11):
        plt.plot(u,Z, \
            color='red', linestyle='solid', linewidth='1.2', \
            marker='o', markersize=4, markeredgewidth=0.8, markerfacecolor = 'white', label = 'T1')


# ------------------------------------------------------------------

def updatetvector(new, index):
    T.remove(T[index])
    T.insert(index,new)
    u=[]
    for i, z in enumerate(Z):
        u.append(Uz(N,z,T[index]))
    ax.lines[index].set_xdata(u)
    fig.canvas.draw()

def updateZ(Nz):
    Z=np.linspace(0.0,2.0,num=Nz)
    for j, t in enumerate(T):
        u=[]
        for i, z in enumerate(Z):
            u.append(Uz(N,z,t))
        ax.lines[j].set_data(u,Z)
    fig.canvas.draw()

def updateN(NewN):
    N=NewN
    for j, t in enumerate(T):
        u=[]
        for i, z in enumerate(Z):
            u.append(Uz(N,z,t))
        ax.lines[j].set_data(u,Z)
    fig.canvas.draw()

Nwidget = widgets.IntSlider(min=1,max=100,step=2,value=19, description=' ', continuous_update=False)
Nzwidget = widgets.IntSlider(min=3, max=100,step=1,value=50, description=' ',disabled=False, continuous_update=False)
Twidget = widgets.FloatSlider(min=0, max=1.5,step=0.1,value=0.5, description=' ',disabled=False, continuous_update=False)
T1widget = widgets.BoundedFloatText(value=0.01, description='T:', disabled=False, step=0.01, min=0.0)
T2widget = widgets.BoundedFloatText(value=0.5, description=' ', disabled=False, step=0.01, min=0)
T3widget = widgets.BoundedFloatText(value=1, description=' ', disabled=False, step=0.01, min=0)

T1widget.layout.width='20%'
T1widget.style.description_width='10%'
T2widget.layout.width='20%'
T2widget.style.description_width='1%'
T3widget.layout.width='20%'
T3widget.style.description_width='1%'

form_item_layout = widgets.Layout(
    display='flex',
    flex_flow='row',
    justify_content='space-between'
)

form_items = [
    widgets.Box([widgets.Label(value='N terms in Fourier series'), Nwidget], layout=form_item_layout),
    widgets.Box([widgets.Label(value='Number of points along depth'), Nzwidget], layout=form_item_layout),
    widgets.Box([widgets.Label(value='Time Factor'), Twidget], layout=form_item_layout),
    widgets.Box([widgets.Label(value='T'),T1widget,T2widget,T3widget], layout=form_item_layout)
]

form = widgets.Box(form_items, layout=widgets.Layout(
    display='flex',
    flex_flow='column',
    border='solid 2px',
    align_items='stretch',
    width='50%'
))
#form

def changeN(change):
    updateN(change['new'])
def changeZ(change):
    updateZ(change['new'])
def updateT(change):
    updatetvector(change['new'],11)
def changeT1(change):
    updatetvector(change['new'],0)
def changeT2(change):
    updatetvector(change['new'],1)
def changeT3(change):
    updatetvector(change['new'],2)

Nwidget.observe(changeN, names='value')
Nzwidget.observe(changeZ, names='value')
Twidget.observe(updateT, names='value')
T1widget.observe(changeT1, names='value')
T2widget.observe(changeT2, names='value')
T3widget.observe(changeT3, names='value')

form


<IPython.core.display.Javascript object>

Box(children=(Box(children=(Label(value='N terms in Fourier series'), IntSlider(value=19, continuous_update=Fa…

You should find that the solutions for 𝑇 larger than about 0.1 are relatively insensitive to 𝑁. However, the number of terms in the Fouier series has a significant effect for low values of 𝑇. This is because the solution at low 𝑇 exhibits a flat region at 𝑈𝑧=0, and significantly more terms are required to obtain an accurate series solution. Using a single term provides a reasonable approximation when 𝑇 is higher than about 0.2.